# Training runs on Google Colab (free tier)

Trains the VGG models used for Figure 1 (with / without BatchNorm) and
Figure 2 (noisy BatchNorm) of Santurkar et al. (2018).

* Results and checkpoints are written to **Google Drive**, so a
  disconnected session loses nothing: re-run the notebook and every
  unfinished run continues from its last checkpoint (`--resume`).
* Finished runs are skipped.
* One 15,000-step run takes roughly 40 minutes on a T4 GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive
A pop-up asks for permission to access your Drive.

In [ ]:
import os
from google.colab import drive

try:
    drive.mount("/content/drive")
except Exception as error:
    print("Drive could not be mounted:", error)

if os.path.ismount("/content/drive"):
    DRIVE_OUT = "/content/drive/MyDrive/BatchNorm_Reproduction/runs"
else:
    # Fallback: results stay on this Colab machine only and are LOST
    # when the session ends. Download them before closing.
    DRIVE_OUT = "/content/runs"
    print("WARNING: Google Drive is NOT mounted. Saving to", DRIVE_OUT,
          "- download the results before the session ends!")

os.makedirs(DRIVE_OUT, exist_ok=True)
print("Results folder:", DRIVE_OUT)

## 3. Get the code
Clones the repository the first time, pulls the latest `main` after that.

In [ ]:
REPO_URL = "https://github.com/Sila402/batch-normalization-reproduction.git"
REPO_DIR = "/content/batch-normalization-reproduction"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q

%cd {REPO_DIR}
!git log --oneline -1

## 4. Choose the runs

Each entry is `(model, learning rate, seed)`. `model` is `with_bn`,
`without_bn` or `noisy_bn`.

Split the list between group members so that each person runs a few
entries per session. Everyone must use the **same** `STEPS`.

In [ ]:
STEPS = 15000

RUNS = [
    ("with_bn",    0.1, 0),
    ("without_bn", 0.1, 0),
    ("noisy_bn",   0.1, 0),
]

## 5. Train
Re-run this cell after a disconnect: finished runs are skipped and unfinished runs resume.

In [ ]:
import os
import csv
import subprocess
import sys
from pathlib import Path


def run_dir(model, lr, seed):
    return Path(DRIVE_OUT) / f"{model}_lr{lr}_seed{seed}"


def finished_steps(model, lr, seed):
    csv_path = run_dir(model, lr, seed) / "metrics.csv"
    if not csv_path.exists():
        return 0
    with open(csv_path, newline="") as f:
        rows = list(csv.DictReader(f))
    return int(rows[-1]["step"]) if rows else 0


# The repository must be cloned (section 3 / "Get the code") first.
if not os.path.isdir(os.path.join(REPO_DIR, "final_reproduction")):
    raise RuntimeError(
        f"Code not found in {REPO_DIR}. Run the 'Get the code' cell "
        "first (or use Runtime/Run all from the top)."
    )
os.chdir(REPO_DIR)

for model, lr, seed in RUNS:
    done = finished_steps(model, lr, seed)
    if done >= STEPS:
        print(f"SKIP {model} lr={lr} seed={seed}: already finished")
        continue

    latest = run_dir(model, lr, seed) / "checkpoints" / "latest.pt"
    command = [
        sys.executable, "-m", "final_reproduction.experiments.train_figure1",
        "--model", model, "--lr", str(lr), "--seed", str(seed),
        "--steps", str(STEPS), "--out", DRIVE_OUT,
    ]
    if latest.exists():
        command.append("--resume")

    print("\nRUN:", " ".join(command[2:]), flush=True)

    # Stream the training output (and any error message) into the
    # notebook. With a plain subprocess.run the child's output does not
    # show up in Colab/Kaggle, so a failure gives no explanation.
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, cwd=REPO_DIR,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(
            f"Training failed for {model} lr={lr} seed={seed}. "
            "See the error message printed above."
        )

## 6. Summary of finished runs
Final test accuracy and the mean of the last 10 evaluations (the last 1,000 steps).

In [ ]:
import pandas as pd

summary = []
for model, lr, seed in RUNS:
    csv_path = run_dir(model, lr, seed) / "metrics.csv"
    if not csv_path.exists():
        continue
    df = pd.read_csv(csv_path)
    summary.append({
        "model": model, "lr": lr, "seed": seed,
        "last_step": int(df["step"].iloc[-1]),
        "final_test_acc": df["test_accuracy"].iloc[-1],
        "mean_last10_test_acc": df["test_accuracy"].tail(10).mean(),
        "final_train_loss": df["train_loss"].iloc[-1],
    })

pd.DataFrame(summary)